# Run GEE Spatial Extractions

Use this notebook as the Colab runner. The reusable workflow code lives in the repo.

Before running the pilot, upload the zipped watershed shapefile listed in `config/gee-assets.yml` to Earth Engine using the asset name in that same config file. The current pilot is `precip`, `2020`, `batch_001`, and it exports to the Drive folder named `Google-Earth-Engine`.

Earth Engine shortens some shapefile column names during upload. This notebook checks for the shortened `run_grp` field before filtering the pilot run.


In [ ]:
REPO_URL = "https://github.com/global-river-chem/data-workflow_spatial.git"
REPO_DIR = "data-workflow_spatial"

import os
import subprocess

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL], check=True)

os.chdir(REPO_DIR)

In [ ]:
!pip -q install -r requirements.txt

In [ ]:
from src.gee_spatial.auth import initialize
from src.gee_spatial.extraction import load_run_config

assets = load_run_config("config/gee-assets.yml")
products = load_run_config("config/driver-products.yml")
run_list = load_run_config("config/run-list.yml")

ee = initialize(project=assets.get("earth_engine", {}).get("project"))

In [ ]:
from src.gee_spatial.datasets import continuous_products, product_names
from src.gee_spatial.extraction import load_watersheds
from src.gee_spatial.runs import choose_property_name

watersheds = load_watersheds(assets["watersheds"]["asset_id"])
configured_run_group_column = assets.get("watersheds", {}).get(
    "earth_engine_run_group_column",
    run_list.get("site_groups", {}).get("column", "run_group"),
)
run_group_column = choose_property_name(
    watersheds,
    configured_run_group_column,
    alternatives=["run_group", "run_grp"],
)
print("Products:", product_names(products))
print("Continuous products:", continuous_products(products))
print("Run group column:", run_group_column)
print("Asset rows:", watersheds.size().getInfo())
print("Run group counts:", watersheds.aggregate_histogram(run_group_column).getInfo())
print(watersheds.limit(1).getInfo())


In [ ]:
from src.gee_spatial.export import export_table_to_drive, task_summary
from src.gee_spatial.extraction import extract_continuous_product
from src.gee_spatial.runs import export_name, filter_watersheds_by_group

pilot = run_list.get("pilot", {})
RUN_EXPORT = bool(pilot.get("launch_export", False))
PRODUCT = pilot.get("product", "precip")
YEAR = pilot.get("year", 2020)
MONTH = pilot.get("month")
RUN_GROUP = pilot.get("run_group", run_list.get("site_groups", {}).get("default", "pilot_01"))

selected_watersheds = filter_watersheds_by_group(
    watersheds,
    run_group=RUN_GROUP,
    run_group_column=run_group_column,
)
selected_row_count = selected_watersheds.size().getInfo()
out_name = export_name(PRODUCT, year=YEAR, month=MONTH, run_group=RUN_GROUP)

print("Planned export:", out_name)
print("Selected rows:", selected_row_count)

if selected_row_count == 0:
    raise ValueError(
        f"No watershed rows selected for {RUN_GROUP} using column {run_group_column}. "
        "Check the uploaded asset column names before exporting."
    )

if RUN_EXPORT:
    export_rows = extract_continuous_product(
        PRODUCT,
        products,
        selected_watersheds,
        year=YEAR,
        month=MONTH,
    )
    task = export_table_to_drive(
        export_rows,
        description=out_name,
        folder=assets["exports"]["drive_folder"],
        file_name_prefix=out_name,
    )
    print(task_summary(task))
else:
    print("Set RUN_EXPORT = True after confirming asset, product, year/month, and run group")
